## Setup

- **API key**: The first code cell loads your OpenRouter API key from a `.env` file. [OpenRouter](https://openrouter.ai) is a service that gives you one place to call many AI models. An API key is a secret code that identifies you to the service. The `.env` values replace any existing environment variables with the same name. If no key is found, the cell asks you to type it in. Never commit your key to Git.
- **Model**: The `MODEL` variable in `.env` chooses the model. If it is not set, the notebook uses `openai/gpt-6-luna`. Model names are listed at [openrouter.ai/models](https://openrouter.ai/models). They can change, so check there if the default stops working.
- **LangSmith**: The notebook loads a prompt called `moche/react` from LangSmith, a service for storing and testing prompts. Set `LANGSMITH_API_KEY` in `.env` before you run that cell.

Your `.env` file needs these variable names (the values are your own secrets):

```
OPENROUTER_API_KEY=
LANGSMITH_API_KEY=
MODEL=            # optional
```

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openai/gpt-6-luna")  # swap on openrouter.ai/models if needed

# **ReAct Prompting Framework Overview**

The ReAct framework was [introduced in a 2022 paper, ReAct: Synergizing Reasoning and Acting in Language Models, by researchers at Google](https://blog.research.google/2022/11/react-synergizing-reasoning-and-acting.html).

### 🌐 **ReAct (Reasoning + Acting) Explained**

- ReAct asks a language model to write its reasoning step by step, and to take actions, such as running a search, between those steps.

- Each action returns an observation (its result). The model reads the observation before deciding what to do next.

### 🔗 **Capabilities of ReAct**

1. 🔄 **Interleaved Reasoning and Acting**: Reasoning and actions alternate, so each new observation can change the plan.

2. 🌍 **External Interactions**: The model can call tools such as a search engine or Wikipedia, instead of relying only on what it learned in training.

3. 🧑‍🎨 **Readable Steps**: The written steps make it easier to follow how the model reached an answer. They are a useful record, but not a guaranteed explanation (see the limitations section at the end).

4. 🛠️ **Varied Tasks**: The paper tested the approach on question answering, fact verification, a text-based household game and online shopping.

<figure>
  <img src="./assets/4_techn.png" alt="Image Description" style="width:100%">
  <figcaption>Comparison of four prompting methods, (a) Standard, (b) Chain-of-thought (CoT, Reason Only), (c) Act-only, and (d) (Reason+Act), solving a HotpotQA question. (2) Comparison of (a) Act-only and (b) prompting methods to solve an interactive AlfWorld task. In both domains, the initial prompts of in-context examples are omitted, and only show task solving trajectories generated by the model (Act, Thought) and the environment (Obs).</figcaption>
  <a href="https://ar5iv.labs.arxiv.org/html/2210.03629">Image Source</a>
</figure>

### 🌟 **Why ReAct Stands Out**

1. **Mixes Reasoning and Acting**: The paper reports gains over reasoning-only and acting-only versions on several of its benchmarks, but not on all of them.

2. **Uses External Data**: Tool results can add specific or recent information the model may not know. The quality of the answer still depends on the tools and sources.

3. **Easier to Inspect**: The written steps show what the model looked up and when, which helps you check its work.

# 🤖 **Overview of ReAct in LangChain**

- [LangChain](https://www.langchain.com) is a library for building apps with language models.

- In this notebook, ReAct is a **prompting style**. The prompt asks the model to write Thought, Action, Action Input and Observation steps.

- The agent is built with `create_agent`. It passes the tools to the model using **native tool calling**: the model returns a structured request to use a tool, and LangChain runs it. The Thought and Action text guides the model, but LangChain does not parse it from the reply.

- Useful for chatbots and question answering. Answer quality depends on the model and the tools.

## 🚀 **Steps to Implement an Agent**

1. ✍️ **Prompt Template Creation**: Load the ReAct prompt from LangSmith and adapt it for the agent.

2. 🧰 **Tool Definition**: Define a web search tool and a Wikipedia tool. Each has a name and a description, which the model reads.

3. 🧠 **LLM Setup**: Choose the model through OpenRouter.

4. 🤹 **Agent Setup**: `create_agent` combines the model, the tools and the system prompt.

5. 🏃 **Agent Usage**: Invoke the agent with a user message, then read the trace it returns.

In [2]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langsmith import Client
from langchain.agents import create_agent 



/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/var/folders/vl/m6g01gp54cl4409b678xyrkm0000gn/T/ipykernel_89237/2870635649.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun


In [3]:
# DuckDuckGo search
search = DuckDuckGoSearchRun()

# Tools are plain functions with @tool. The docstring is the description the agent sees.
# Tool names must match pattern: ^[a-zA-Z0-9_-]+$ (no spaces)
@tool("current_event_search")
def search_tool(query: str) -> str:
    """Useful for when you need to search about current events."""
    return search.run(query)

# Wikipedia
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=None))

@tool("noun_search")
def wiki_tool(query: str) -> str:
    """Useful for when you need to search for information about nouns."""
    return wiki.run(query)

tools = [search_tool, wiki_tool]

## Loading the Prompt from LangSmith

A **prompt** is the set of written instructions that shapes how a model behaves. Keeping the prompt in LangSmith, rather than in the code, lets you edit and test it separately. The notebook downloads the prompt named `moche/react`. If someone changes that prompt on LangSmith, the agent's behaviour can change without any change to this notebook.

The code passes `dangerously_pull_public_prompt=True` because the current LangSmith library asks for it on owner/name pulls. Keep it as written, and check the LangSmith documentation before you change it.

In [4]:

client = Client()

# This is your own prompt on LangSmith. langsmith 0.14+ requires the flag below for any owner/name pull.
system_prompt = client.pull_prompt("moche/react", dangerously_pull_public_prompt=True)

In [5]:
print(system_prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [6]:
# View the template structure
print("Template variables:", system_prompt.input_variables)
print("\nTemplate:")
print(system_prompt.template)

# The next cell turns this template into the agent's system prompt.
# {tools} and {tool_names} are filled in from the tools in cell 3.
# {input} and {agent_scratchpad} are removed, because create_agent takes the question as a message and manages the scratchpad.

Template variables: ['agent_scratchpad', 'input', 'tool_names', 'tools']

Template:
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


## Understanding the Prompt Template

The template contains **placeholders**: gaps in curly braces that the code fills in later.

- `{tools}`: replaced with each tool's name and description.
- `{tool_names}`: replaced with a comma-separated list of the tool names.
- `{input}`: the user's question. It is removed, because `create_agent` takes the question as a chat message instead.
- `{agent_scratchpad}`: a running record of the Thought, Action and Observation steps so far. It is removed, because `create_agent` manages this record itself.

The next cells fill in the first two placeholders and remove the last two.

In [7]:
llm = ChatOpenRouter(model=MODEL, temperature=0.0)

In [8]:

# Build the ReAct system prompt from the pulled template (option b)
# - {tools} gets each tool's name and description
# - {tool_names} gets the comma-separated tool names
# - {input} and {agent_scratchpad} are removed, because create_agent handles them
tool_descriptions = "\n".join(f"{t.name}: {t.description}" for t in tools)
tool_names = ", ".join(t.name for t in tools)

react_system_prompt = (
    system_prompt.template
    .replace("{tools}", tool_descriptions)
    .replace("{tool_names}", tool_names)
    .replace("{input}", "")
    .replace("{agent_scratchpad}", "")
)

agent = create_agent(
    model=llm, 
    tools=tools, 
    system_prompt=react_system_prompt
)

## Worked Example: One Thought, Action, Observation Loop

This example is made up to show the pattern. It is **not** output from this notebook, and a real run will look different.

```
Question: What is the name of the cafe on the third floor of the Example Building?
Thought: I need information about a specific building. I will search Wikipedia.
Action: noun_search
Action Input: Example Building cafe
Observation: No results found.
Thought: The search found nothing, so I should not guess a name. I now know the final answer.
Final Answer: I could not find information about a cafe in the Example Building.
```

Points to notice:

- The **Observation** is the only outside information the model gets. Everything else comes from the model itself.
- The model can say it does not know. The prompt does not guarantee this, so you still need to check the answers.
- The **Thought** lines are text the model writes. They show its plan, but they are not a guaranteed record of why it acted.

## Reading the Agent's Output

`agent.invoke` returns a dictionary with one key, `messages`. This is a list of messages in the order they happened:

- **HumanMessage**: your question.
- **AIMessage with tool calls**: the model's request to run a tool, with the tool name and input. Its `content` can be empty when the model is only asking for a tool.
- **ToolMessage**: the result of the tool. This is the observation.
- The request and result pair can repeat several times.
- **Final AIMessage**: the answer, with no further tool request.

Some messages also carry a `reasoning_content` field. This is extra information that the model provider returns, and it depends on the provider. You can ignore it for this lesson.

The printed output can be long. Scroll through it and match each step to the Thought, Action and Observation pattern above.

In [9]:
agent.invoke({"messages": [{"role": "user", "content": "Who is playing in the Super Bowl in 2025? Where will it be played?"}]})

{'messages': [HumanMessage(content='Who is playing in the Super Bowl in 2025? Where will it be played?', additional_kwargs={}, response_metadata={}, id='de81bff6-d355-471d-bf45-97935d9d9c3b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': '**Searching for current answers**\n\nI need to address a user query regarding a current answer from 2025. It seems the approach is to search for relevant information using a tool and provide a formatted response. I should think about how best to display the findings, possibly using a commentary format for clarity. Let’s get started on this search to pull the most accurate information needed for response!', 'reasoning_details': [{'summary': '**Searching for current answers**\n\nI need to address a user query regarding a current answer from 2025. It seems the approach is to search for relevant information using a tool and provide a formatted response. I should think about how best to display the findings, possibly using a commentary f

## Limitations and Responsible Use

- **Cost and speed**: Each step is another call to the model, and each tool call takes time. A question that needs several searches can be slower and more expensive than a single model call.
- **Tool failures**: A search can fail or return nothing, as in the example above. The agent may then guess or give a weak answer.
- **Stale or biased sources**: Search results and Wikipedia can be out of date, incomplete or one-sided.
- **Confident wrong answers**: A Final Answer can sound certain even when the observations do not support it.
- **Prompt injection**: A web page can contain text written to manipulate an AI model, such as "ignore your instructions". The agent reads tool results as input, so this is a risk when it searches the open web.
- **Privacy**: Your question and the search queries leave your computer. They are sent through OpenRouter to the model provider, and to the search and Wikipedia services. Do not include personal or confidential information.
- **Responsible use**: Follow your course's rules on AI use, and say when you have used AI in your work.

### Verify before you trust

- [ ] Did the agent use a tool, and did that tool return relevant results?
- [ ] Does the Final Answer match what the observations say?
- [ ] Can you find the key claim in a source you trust?
- [ ] Does your question include information you should not share?